# Devoir 1 — ETE435 Géoinformatique
## Ingénierie des données géospatiales : fiabilité, distances et proximité

**Cours :** ETE435 — Géoinformatique | **Session :** H2026


Ce notebook présente notre démarche complète pour le traitement du jeu de données des aéroports canadiens — du nettoyage jusqu'à l'export géospatial.

**Note sur l'aide externe :** Nous avons utilisé Claude de chez (Anthropic) comme outil d'assistance pour la rédaction et la vérification du code. L'exécution, l'interprétation des résultats et les réponses aux questions de réflexion sont les nôtres.

## Partie 0 — Environnement de travail

Pour que le code puisse tourner sur n'importe quelle machine sans problème de compatibilité, on a créé un environnement virtuel Python dédié (`venv_devoir2`). Cela isole les bibliothèques du projet du reste du système.

```bash
python -m venv venv_devoir2
venv_devoir2/Scripts/python -m pip install pandas geopandas shapely notebook
venv_devoir2/Scripts/python -m pip freeze > requirements.txt
```

Le fichier `requirements.txt` (joint à la remise) documente toutes les dépendances avec leurs versions exactes utilise dans ce projet.

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import math
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import platform

# ── Versions (traçabilité — Question de réflexion 4) ─────────────────────────
print(f"Python    : {platform.python_version()}")
print(f"pandas    : {pd.__version__}")
print(f"geopandas : {gpd.__version__}")
import shapely
print(f"shapely   : {shapely.__version__}")
print(f"OS        : {platform.system()} {platform.release()} ({platform.machine()})")

Python    : 3.13.14
pandas    : 3.0.3
geopandas : 1.1.4
shapely   : 2.1.2
OS        : Windows 11 (AMD64)


## Partie 1 — Audit et nettoyage des données

Le fichier CSV brut ne peut pas être utilisé tel quel : plusieurs lignes ont des coordonnées manquantes et d'autres contiennent des valeurs qui ne devraient pas être là (par exemple, une longitude qui vaut `"ETE435"` — le code du cours au lieu d'un nombre).

On procède en trois étapes : d'abord repérer les valeurs textuelles avant conversion, ensuite convertir les colonnes en numérique avec `errors='coerce'` (les valeurs non convertibles deviennent `NaN`), et finalement supprimer les lignes incomplètes ainsi que les coordonnées géographiquement impossibles pour le Canada.

In [2]:
# ── 1. Chargement et audit initial ───────────────────────────────────────────
FICHIER_CSV = "ETE_435_Donnees_Devoir_1_H26 (1).csv"

df_brut = pd.read_csv(FICHIER_CSV, low_memory=False)

n_initial = len(df_brut)
print(f"Nombre de lignes initiales : {n_initial}")
print(f"Nombre de colonnes         : {len(df_brut.columns)}")
print(f"\nColonnes disponibles :")
print(df_brut.columns.tolist())
print(f"\nTypes de données :")
print(df_brut[["ident", "type", "name", "latitude_deg", "longitude_deg"]].dtypes)
print(f"\nAperçu des 5 premières lignes :")
df_brut[["ident", "name", "type", "latitude_deg", "longitude_deg"]].head()

Nombre de lignes initiales : 417
Nombre de colonnes         : 24

Colonnes disponibles :
['id', 'ident', 'type', 'name', 'latitude_deg', 'longitude_deg', 'elevation_ft', 'continent', 'country_name', 'iso_country', 'region_name', 'iso_region', 'local_region', 'municipality', 'scheduled_service', 'gps_code', 'icao_code', 'iata_code', 'local_code', 'home_link', 'wikipedia_link', 'keywords', 'score', 'last_updated']

Types de données :
ident                str
type                 str
name                 str
latitude_deg     float64
longitude_deg        str
dtype: object

Aperçu des 5 premières lignes :


,ident,name,type,latitude_deg,longitude_deg
0,CYYZ,Toronto Pearson International Airport,large_airport,43.675935,-79.629421
1,CYVR,Vancouver International Airport,large_airport,49.193901,-123.183998
2,CYUL,Montreal / Pierre Elliott Trudeau Internationa...,large_airport,45.467837,-73.742294
3,CYYC,Calgary International Airport,large_airport,51.118822,-114.009933
4,CYOW,Ottawa Macdonald-Cartier International Airport,large_airport,45.322498,-75.669197


In [3]:
# ── 2a. Copie de travail + détection des valeurs NON NUMÉRIQUES ──────────────
# On travaille sur une copie pour ne pas modifier le DataFrame brut original
df = df_brut.copy()

# Identifier les cellules non vides mais non convertibles en float
# AVANT to_numeric — permet de les signaler séparément dans le rapport
non_num_lat = df[pd.to_numeric(df["latitude_deg"],  errors="coerce").isna()
                 & df["latitude_deg"].notna()]
non_num_lon = df[pd.to_numeric(df["longitude_deg"], errors="coerce").isna()
                 & df["longitude_deg"].notna()]

print(f"Valeurs non numériques dans latitude_deg  : {len(non_num_lat)}")
print(f"Valeurs non numériques dans longitude_deg : {len(non_num_lon)}")
if len(non_num_lon) > 0:
    print("\nDétail :")
    print(non_num_lon[["ident", "name", "latitude_deg", "longitude_deg"]].to_string(index=False))

# ── 2b. Conversion numérique robuste ─────────────────────────────────────────
# errors='coerce' : texte parasite ou vide → NaN (ne plante pas le script)
df["latitude_deg"]  = pd.to_numeric(df["latitude_deg"],  errors="coerce")
df["longitude_deg"] = pd.to_numeric(df["longitude_deg"], errors="coerce")

print("\nTypes après conversion :")
print(df[["latitude_deg", "longitude_deg"]].dtypes)
print(f"\nNaN latitude  : {df['latitude_deg'].isna().sum()}")
print(f"NaN longitude : {df['longitude_deg'].isna().sum()}")

Valeurs non numériques dans latitude_deg  : 0
Valeurs non numériques dans longitude_deg : 2

Détail :
ident                                  name  latitude_deg longitude_deg
 CEL5                    Valleyview Airport     55.032941         CEL15
 CAZ5 Cache Creek-Ashcroft Regional Airport     50.775258        ETE435

Types après conversion :
latitude_deg     float64
longitude_deg    float64
dtype: object

NaN latitude  : 37
NaN longitude : 39


In [4]:
# ── 3. Suppression des lignes avec coordonnées manquantes ────────────────────
masque_manquant = df["latitude_deg"].isna() | df["longitude_deg"].isna()
lignes_manquantes = df[masque_manquant][["ident", "name", "latitude_deg", "longitude_deg"]]

n_manquantes = masque_manquant.sum()
print(f"Lignes supprimées (coordonnées manquantes) : {n_manquantes}")
print("\nDétail des lignes rejetées :")
print(lignes_manquantes.to_string(index=False))

df = df[~masque_manquant].reset_index(drop=True)
print(f"\nLignes restantes après suppression : {len(df)}")

Lignes supprimées (coordonnées manquantes) : 58

Détail des lignes rejetées :
  ident                                                        name  latitude_deg  longitude_deg
   CYWG Winnipeg / James Armstrong Richardson International Airport     49.910000            NaN
   CYYJ                              Victoria International Airport     48.647201            NaN
   CYLW                               Kelowna International Airport     49.956100            NaN
   CYXE         Saskatoon John G. Diefenbaker International Airport           NaN    -106.700793
CA-1108                               Buttonville Municipal Airport           NaN     -79.370003
   CYXX                            Abbotsford International Airport           NaN    -122.361000
   CYFC                           Fredericton International Airport           NaN     -66.529891
   CYFD                                 Brantford Municipal Airport     43.131401            NaN
   CYKA           Kamloops John Moose Fulton Fiel

In [5]:
# ── 4. Détection et suppression des valeurs aberrantes ───────────────────────
# Bornes géographiques raisonnables pour le Canada :
#   latitude  : [42°N, 85°N]   (du lac Érié à l'île d'Ellesmere)
#   longitude : [-142°E, -52°E] (de la frontière Alaska/Yukon à Terre-Neuve)
LAT_MIN, LAT_MAX   = 42.0,  85.0
LON_MIN, LON_MAX   = -142.0, -52.0

masque_aberrant = (
    (df["latitude_deg"]  < LAT_MIN) | (df["latitude_deg"]  > LAT_MAX) |
    (df["longitude_deg"] < LON_MIN) | (df["longitude_deg"] > LON_MAX)
)

lignes_aberrantes = df[masque_aberrant][["ident", "name", "latitude_deg", "longitude_deg"]]
n_aberrantes = masque_aberrant.sum()

print(f"Lignes avec valeurs aberrantes : {n_aberrantes}")
if n_aberrantes > 0:
    print("\nDétail :")
    print(lignes_aberrantes.to_string(index=False))
else:
    print("Aucune valeur aberrante détectée en dehors des bornes canadiennes.")

df = df[~masque_aberrant].reset_index(drop=True)

Lignes avec valeurs aberrantes : 3

Détail :
ident                 name  latitude_deg  longitude_deg
 CYPT Pelee Island Airport     41.778033     -82.674222
 CYFH    Fort Hope Airport     51.561901    -150.000000
 CYGZ  Grise Fiord Airport     92.567680     -82.908583


In [6]:
# ── 5. Résumé du protocole de nettoyage ──────────────────────────────────────
n_valides = len(df)
n_supprimes_total = n_initial - n_valides

print("=" * 55)
print("RÉSUMÉ DU PROTOCOLE DE NETTOYAGE")
print("=" * 55)
print(f"  Lignes initiales                : {n_initial:>4}")
print(f"  Supprimées (coord. manquantes)  : {n_manquantes:>4}")
print(f"  Supprimées (valeurs aberrantes) : {n_aberrantes:>4}")
print(f"  ─────────────────────────────────────")
print(f"  Lignes valides restantes        : {n_valides:>4}")
print("=" * 55)

RÉSUMÉ DU PROTOCOLE DE NETTOYAGE
  Lignes initiales                :  417
  Supprimées (coord. manquantes)  :   58
  Supprimées (valeurs aberrantes) :    3
  ─────────────────────────────────────
  Lignes valides restantes        :  356


In [7]:
# ── 6. Export du fichier nettoyé ─────────────────────────────────────────────
df.to_csv("airports_clean.csv", index=False)
print("Fichier 'airports_clean.csv' exporté avec succès.")

Fichier 'airports_clean.csv' exporté avec succès.


## Partie 2 — Distance de Haversine

On ne peut pas simplement calculer `sqrt((lat2-lat1)² + (lon2-lon1)²)` en degrés pour mesurer une distance entre aéroports. Le problème, c'est que la Terre est ronde et qu'un degré de longitude ne représente pas la même distance selon la latitude où on se trouve. À Montréal (45°N), un degré de longitude vaut environ 79 km. À Whitehorse (60°N), ce même degré ne vaut plus que 55 km. La formule euclidienne ignorerait complètement cette distorsion.

La formule de Haversine résout ça en calculant la distance sur la surface de la sphère terrestre (arc de grand cercle) :

$$a = \sin^2\!\left(\frac{\Delta\phi}{2}\right) + \cos(\phi_1)\cdot\cos(\phi_2)\cdot\sin^2\!\left(\frac{\Delta\lambda}{2}\right)$$
$$d = 2R \cdot \arctan2\!\left(\sqrt{a},\, \sqrt{1-a}\right)$$

où φ est la latitude en radians, λ la longitude, et R = 6 371 km. On utilise `arctan2` plutôt qu'`arcsin` parce qu'il est numériquement plus stable pour toutes les distances.

In [8]:
# ── Fonction Haversine ────────────────────────────────────────────────────────
def calculer_haversine(lat1, lon1, lat2, lon2):
    """
    Retourne la distance en km entre deux points géographiques
    en utilisant la formule de Haversine.

    Paramètres : lat1, lon1, lat2, lon2 en degrés décimaux.
    Retourne   : distance en kilomètres (float), ou None si entrée invalide.

    Formule :
        a = sin²(Δφ/2) + cos(φ₁)·cos(φ₂)·sin²(Δλ/2)
        d = 2R · arctan2(√a, √(1−a))
    avec R = 6 371 km (rayon moyen de la Terre).
    """
    try:
        lat1 = float(lat1)
        lon1 = float(lon1)
        lat2 = float(lat2)
        lon2 = float(lon2)
    except (TypeError, ValueError):
        print(f"[ERREUR] Entrée non numérique : ({lat1}, {lon1}, {lat2}, {lon2})")
        return None

    R = 6371.0  # rayon moyen de la Terre en km

    # Conversion degrés → radians
    phi1    = math.radians(lat1)
    phi2    = math.radians(lat2)
    dphi    = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)

    # Terme intermédiaire a
    a = (math.sin(dphi / 2) ** 2
         + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2) ** 2)

    # atan2 est plus stable numériquement que asin pour les grandes distances
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

In [9]:
# ── Validation : distance CYYZ (Toronto Pearson) → CYVR (Vancouver) ──────────
def obtenir_coordonnees(df, ident):
    """Retourne (lat, lon) d'un aéroport à partir de son identifiant."""
    ligne = df[df["ident"] == ident]
    if ligne.empty:
        raise ValueError(f"Aéroport '{ident}' introuvable dans le DataFrame.")
    return float(ligne["latitude_deg"].values[0]), float(ligne["longitude_deg"].values[0])

try:
    lat_yyz, lon_yyz = obtenir_coordonnees(df, "CYYZ")
    lat_yvr, lon_yvr = obtenir_coordonnees(df, "CYVR")

    distance_yyz_yvr = calculer_haversine(lat_yyz, lon_yyz, lat_yvr, lon_yvr)

    print(f"Toronto Pearson (CYYZ) : lat={lat_yyz:.6f}, lon={lon_yyz:.6f}")
    print(f"Vancouver Intl  (CYVR) : lat={lat_yvr:.6f}, lon={lon_yvr:.6f}")
    print(f"\nDistance CYYZ → CYVR  : {distance_yyz_yvr:.2f} km")
    print("(Valeur attendue : ~3 360 km selon les références géographiques)")
except ValueError as e:
    print(f"[ERREUR] {e}")

# Test de robustesse : entrée non numérique
print("\nTest robustesse (entrée invalide) :")
resultat = calculer_haversine("abc", None, 49.19, -123.18)
print(f"  Résultat : {resultat}")

Toronto Pearson (CYYZ) : lat=43.675935, lon=-79.629421
Vancouver Intl  (CYVR) : lat=49.193901, lon=-123.183998

Distance CYYZ → CYVR  : 3345.66 km
(Valeur attendue : ~3 360 km selon les références géographiques)

Test robustesse (entrée invalide) :
[ERREUR] Entrée non numérique : (abc, None, 49.19, -123.18)
  Résultat : None


## Partie 3 — Analyse de proximité et export géospatial

On filtre les `large_airport`, on calcule la distance de chacun à Montréal-Trudeau (CYUL), puis on exporte le tout en GeoJSON avec le CRS EPSG:4326 (WGS 84 — le standard en degrés lat/lon, compatible avec QGIS, ArcGIS et la plupart des SIG).

In [10]:
# ── 12. Filtrage : large_airport uniquement ───────────────────────────────────
df_large = df[df["type"] == "large_airport"].copy().reset_index(drop=True)
print(f"Nombre de 'large_airport' dans le fichier nettoyé : {len(df_large)}")
print(df_large[["ident", "name", "latitude_deg", "longitude_deg"]].to_string(index=False))

Nombre de 'large_airport' dans le fichier nettoyé : 24
ident                                                    name  latitude_deg  longitude_deg
 CYYZ                   Toronto Pearson International Airport     43.675935     -79.629421
 CYVR                         Vancouver International Airport     49.193901    -123.183998
 CYUL Montreal / Pierre Elliott Trudeau International Airport     45.467837     -73.742294
 CYYC                           Calgary International Airport     51.118822    -114.009933
 CYOW          Ottawa Macdonald-Cartier International Airport     45.322498     -75.669197
 CYEG                          Edmonton International Airport     53.309700    -113.580002
 CYHZ               Halifax / Stanfield International Airport     44.880798     -63.508598
 CYQB                Quebec Jean Lesage International Airport     46.791100     -71.393303
 CYXU                            London International Airport     43.032845     -81.149003
 CYHM            John C. Munro Hami

In [11]:
# ── 13. Calcul de la distance à CYUL (Montréal-Trudeau) ─────────────────────
try:
    lat_cyul, lon_cyul = obtenir_coordonnees(df, "CYUL")
    print(f"Référence CYUL : lat={lat_cyul:.6f}, lon={lon_cyul:.6f}")
except ValueError as e:
    print(f"[ERREUR] {e}")
    raise  # on arrête si CYUL est introuvable

# Calcul vectorisé : on applique la fonction ligne par ligne
df_large["dist_to_CYUL_km"] = df_large.apply(
    lambda row: calculer_haversine(
        row["latitude_deg"], row["longitude_deg"],
        lat_cyul, lon_cyul
    ),
    axis=1
)

# Tri décroissant pour voir le plus éloigné en premier
df_large_trie = df_large[["ident", "name", "latitude_deg", "longitude_deg", "dist_to_CYUL_km"]]\
    .sort_values("dist_to_CYUL_km", ascending=False)

print("\nGrands aéroports triés par distance à CYUL (du plus éloigné au plus proche) :")
print(df_large_trie.to_string(index=False))

Référence CYUL : lat=45.467837, lon=-73.742294

Grands aéroports triés par distance à CYUL (du plus éloigné au plus proche) :
ident                                                    name  latitude_deg  longitude_deg  dist_to_CYUL_km
 CYXY         Whitehorse / Erik Nielsen International Airport     60.708533    -135.065705      4242.433305
 CYQQ          Comox Valley International Airport / CFB Comox     49.710800    -124.887001      3789.415827
 CYVR                         Vancouver International Airport     49.193901    -123.183998      3682.166441
 CYXS                            Prince George (Intl) Airport     53.884311    -122.666554      3565.186915
 CYZF                       Yellowknife International Airport     62.462799    -114.440002      3181.036893
 CYYC                           Calgary International Airport     51.118822    -114.009933      3004.831015
 CYEG                          Edmonton International Airport     53.309700    -113.580002      2968.827501
 CYQR     

In [12]:
# ── 14-15. Création du GeoDataFrame avec CRS EPSG:4326 ───────────────────────
# Point(longitude, latitude) — convention GeoJSON : lon d'abord, lat ensuite
geometries = [
    Point(row["longitude_deg"], row["latitude_deg"])
    for _, row in df_large.iterrows()
]

gdf = gpd.GeoDataFrame(df_large, geometry=geometries, crs="EPSG:4326")

print(f"GeoDataFrame créé — {len(gdf)} entités")
print(f"CRS : {gdf.crs}")
print(f"\nAperçu :")
gdf[["ident", "name", "dist_to_CYUL_km", "geometry"]].head()

GeoDataFrame créé — 24 entités
CRS : EPSG:4326

Aperçu :


,ident,name,dist_to_CYUL_km,geometry
0,CYYZ,Toronto Pearson International Airport,506.953392,POINT (-79.62942 43.67594)
1,CYVR,Vancouver International Airport,3682.166441,POINT (-123.184 49.1939)
2,CYUL,Montreal / Pierre Elliott Trudeau Internationa...,0.000000,POINT (-73.74229 45.46784)
3,CYYC,Calgary International Airport,3004.831015,POINT (-114.00993 51.11882)
4,CYOW,Ottawa Macdonald-Cartier International Airport,151.319185,POINT (-75.6692 45.3225)


In [13]:
# ── 16. Export au format GeoJSON ──────────────────────────────────────────────
FICHIER_SORTIE = "large_airports_canada.geojson"

# Sélection des colonnes à exporter
colonnes_export = ["ident", "name", "type", "municipality", "iso_region",
                   "latitude_deg", "longitude_deg", "dist_to_CYUL_km", "geometry"]
colonnes_export = [c for c in colonnes_export if c in gdf.columns]

gdf[colonnes_export].to_file(FICHIER_SORTIE, driver="GeoJSON")
print(f"Fichier '{FICHIER_SORTIE}' exporté avec succès.")
print(f"CRS du fichier exporté : EPSG:4326")

Fichier 'large_airports_canada.geojson' exporté avec succès.
CRS du fichier exporté : EPSG:4326


In [14]:
# ── Résultat final : aéroport le plus éloigné de CYUL ────────────────────────
plus_eloigne = df_large_trie.iloc[0]
print("AÉROPORT LE PLUS ÉLOIGNÉ DE CYUL (Montréal-Trudeau) :")
print(f"  Ident   : {plus_eloigne['ident']}")
print(f"  Nom     : {plus_eloigne['name']}")
print(f"  Distance: {plus_eloigne['dist_to_CYUL_km']:.2f} km")

AÉROPORT LE PLUS ÉLOIGNÉ DE CYUL (Montréal-Trudeau) :
  Ident   : CYXY
  Nom     : Whitehorse / Erik Nielsen International Airport
  Distance: 4242.43 km


## Questions de réflexion

### Question 1 — Anomalies identifiées

**CAZ5 — Cache Creek-Ashcroft Regional Airport**

Ce qui a attiré notre attention sur cette ligne, c'est que la colonne `longitude_deg` contenait la chaîne `"ETE435"` — le nom du cours — au lieu d'une valeur numérique. C'est une erreur de saisie évidente dans le fichier source. Lors du nettoyage, on a d'abord détecté cette anomalie avant la conversion (`pd.to_numeric` avec `errors='coerce'`), qui l'a transformée en `NaN`, puis la ligne a été supprimée avec toutes les autres coordonnées manquantes.

**CYGZ — Grise Fiord Airport**

La latitude enregistrée est 92,57°N. C'est mathématiquement impossible — la latitude maximale sur Terre est 90°N (le pôle Nord). Cette valeur dépasse de plus de 8 degrés la limite physique absolue. L'aéroport de Grise Fiord se trouve réellement à environ 76,4°N dans le Nunavut. La ligne a été rejetée par notre filtre `latitude_deg > 84`.

---

### Question 2 — Pourquoi Haversine et pas une distance euclidienne ?

La distance euclidienne appliquée directement aux degrés produit des résultats erronés pour trois raisons.

Premièrement, la Terre est une sphère. La distance réelle entre deux points se mesure le long de la surface, pas en ligne droite à travers le globe. L'erreur est négligeable pour des distances courtes, mais pour Montréal–Whitehorse (~4 242 km), elle atteindrait plusieurs centaines de kilomètres.

Deuxièmement, les degrés de longitude ne correspondent pas à la même distance selon la latitude. À l'équateur, 1° de longitude ≈ 111 km. À Montréal (45°N), c'est environ 79 km. À Whitehorse (60°N), plus que 55 km. Utiliser des degrés bruts dans une formule euclidienne, c'est comparer des grandeurs incomparables — l'erreur s'accumule surtout dans le sens est-ouest.

Troisièmement, il n'y a pas d'unité physique dans le résultat. `sqrt((Δlat)² + (Δlon)²)` donne des "degrés composites" qu'on ne peut pas directement convertir en kilomètres sans tenir compte de la latitude — ce que fait précisément Haversine.

---

### Question 3 — L'aéroport le plus éloigné de CYUL

Selon nos calculs, c'est **CYXY — Whitehorse / Erik Nielsen International Airport** avec **4 242,43 km**.

Ce résultat est cohérent géographiquement. Montréal est à l'est du Canada (45°N, 74°O). Whitehorse est au nord-ouest dans le Yukon (61°N, 135°O) — l'écart longitudinal d'environ 61 degrés combiné à un décalage latitudinal de 15 degrés explique cette distance maximale. Les quatre autres aéroports du classement (Comox, Vancouver, Prince George, Yellowknife) sont tous en Colombie-Britannique, au Yukon ou dans les Territoires du Nord-Ouest, ce qui confirme que les `large_airport` les plus éloignés de Montréal se trouvent tous à l'ouest et au nord-ouest du pays.

---

### Question 4 — Traçabilité de l'environnement

| Composant | Version |
|-----------|---------|
| Python | 3.13.14 |
| pandas | 3.0.3 |
| geopandas | 1.1.4 |
| shapely | 2.1.2 |
| numpy | 2.5.0 |
| pyproj | 3.7.2 |
| pyogrio | 0.13.0 |
| notebook | 7.6.0 |
| OS | Windows 11 (64-bit) |

Toutes les dépendances sont consignées dans `requirements.txt`, accessible sur le dépôt GitHub : https://github.com/T-kitt-dan/devoir/blob/main/requirements.txt